# Custom MCP Client: Minimal Example

Below is a simplified Python client example.

The purpose is to connect to the open-source filesystem MCP server, list available tools, and call one tool.

# `@modelcontextprotocol/server-filesystem`
This is an official Model Context Protocol (MCP) server published by Anthropic that exposes your file system as MCP tools.

## What It Does
It starts a Node.js process that:

- Listens on stdio for MCP client connections
- Exposes file system operations as standardized tools
- Serves files from a specified directory (the one you pass in)
- Allows AI agents (like Claude) to read, write, and navigate files safely

## How It Works
```sh
Your Python Client (Jupyter)
    ↓
    └─ spawns: npx @modelcontextprotocol/server-filesystem /path/to/serve
         ↓
         └─ Node.js process starts
         └─ Listens on stdin/stdout for MCP protocol messages
         └─ Your Python client connects via stdio
         ↓
    → Your client can now call file system tools
```
## Available Tools
The filesystem server exposes these MCP tools:

|Tool|	What it does|
|--|--|
|read_file	|Read contents of a file|
|write_file	|Write/create a file|
|list_directory	|List files in a directory|
|move_file	|Move/rename a file|
|delete_file|	Delete a file|
|create_directory	|Create a new directory|
|get_file_stats	|Get file metadata (size, permissions, etc.)|

## Example Use Cases
1. Claude asks to read a log file:
```sh
Claude → MCP Client → filesystem server → reads /path/to/logfile.txt → returns content
```
2. Claude wants to list files in a directory:
```sh
Claude → MCP Client → filesystem server → lists /path/to/directory → returns file list
```
3. Claude wants to create a Python script:
```sh
Claude → MCP Client → filesystem server → writes /path/to/script.py → file created
```
## Security Boundary
Important: The server only has access to files inside the directory you specify.

For example:
```sh
npx @modelcontextprotocol/server-filesystem /Users/tripathimachine/Desktop/GitHub_Repo/ai-agent-docker-poc/MCP_Client
```
This means:

- ✅ Can access /Users/.../MCP_Client/file.txt
- ✅ Can access /Users/.../MCP_Client/subfolder/file.txt
- ❌ Cannot access /Users/.../Desktop/other_file.txt (outside the boundary)

Code snippet
```sh
serve_dir = "../MCP_Client"  # ← The filesystem server will only see files here

StdioServerParameters(
    command="npx",
    args=[
        "-y",
        "@modelcontextprotocol/server-filesystem",
        serve_dir
    ],
)
```
This tells the server: "Start a file system service for the ../MCP_Client directory"

Then your client can call:
```sh
result = await session.call_tool(
    "list_directory",
    arguments={"path": "."}  # "." means the root serve_dir
)
```

## Why Use It?
1. Give AI agents file access — Claude can read your project, modify files, etc.
2. Standardized interface — Uses MCP protocol instead of custom APIs
3. Safe sandbox — Restricted to one directory
4. Real-world example — Great for understanding how MCP servers work

## In Practice
This is how Claude Desktop or other AI tools can:

- ✅ Read your code to provide insights
- ✅ Generate new files for your project
- ✅ Modify configuration files
- ✅ Create test files
- ✅ Explore your project structure

All while being sandboxed to a specific directory for security.

In [ ]:
import asyncio
import os
from pathlib import Path

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


async def main():
    # Use absolute path instead of relative
    # serve_dir = str(Path("/Users/tripathimachine/Desktop/GitHub_Repo/ai-agent-docker-poc/MCP_Client").absolute())
    # serve_dir = os.path.expanduser(serve_dir)  # Expand ~ if present
    serve_dir = "../MCP_Client"
    print(f"[DEBUG] Serving directory: {serve_dir}")
    print(f"[DEBUG] Directory exists: {os.path.isdir(serve_dir)}")
    
    server_params = StdioServerParameters(
        command="npx",
        args=[
            "-y",
            "@modelcontextprotocol/server-filesystem",
            serve_dir
        ],
    )

    try:
        print("[DEBUG] Starting MCP server connection...")
        async with stdio_client(server_params) as (read, write):
            print("[DEBUG] Connected! Initializing session...")
            async with ClientSession(read, write) as session:
                print("[DEBUG] Initializing...")
                await session.initialize()
                print("[DEBUG] ✓ Session initialized!")

                print("\n[DEBUG] Listing tools...")
                tools = await session.list_tools()
                print(f"[DEBUG] Got {len(tools.tools)} tools")

                print("\nAvailable tools:")
                for tool in tools.tools:
                    print(f"  - {tool.name}: {tool.description}")

                print("\n[DEBUG] Calling list_directory...")
                result = await session.call_tool(
                    "list_directory",
                    arguments={"path": "."}
                )

                print("\nTool result:")
                print(result)
    except Exception as e:
        print(f"[ERROR] {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()

# if __name__ == "__main__":
#     asyncio.run(main())

In [28]:
await main()

[DEBUG] Working directory: /Users/tripathimachine/Desktop/GitHub_Repo/ai-agent-docker-poc/MCP_Client
[DEBUG] Serving directory: /Users/tripathimachine/Desktop/GitHub_Repo/ai-agent-docker-poc/mcp-demo-files
[DEBUG] Directory exists: True
[DEBUG] Starting MCP server connection...
[DEBUG] Connected! Initializing session...
[DEBUG] Initializing...
[DEBUG] ✓ Session initialized!

[DEBUG] Listing tools...
[DEBUG] Got 14 tools

Available tools:
  - read_file: Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.
  - read_text_file: Read the complete contents of a file from the file system as text. Handles various text encodings and provides detailed error messages if the file cannot be read. Use this tool when you need to examine the contents of a single file. Use the 'head' parameter to read only the first N lines of a file, or the 'tail' parameter to read only the last N lines of a file. Operates on the file as text regardless of extension. Only works within 

```python
import asyncio
import os
from pathlib import Path

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


async def main():
    # Use an absolute path to avoid typos and relative-path issues.
    serve_dir = (Path.cwd() / ".." / "mcp-demo-files").resolve()
    print(f"[DEBUG] Working directory: {Path.cwd()}")
    print(f"[DEBUG] Serving directory: {serve_dir}")
    print(f"[DEBUG] Directory exists: {serve_dir.is_dir()}")

    server_params = StdioServerParameters(
        command="npx",
        args=[
            "-y",
            "@modelcontextprotocol/server-filesystem",
            str(serve_dir),
        ],
    )

    try:
        print("[DEBUG] Starting MCP server connection...")
        async with stdio_client(server_params) as (read, write):
            print("[DEBUG] Connected! Initializing session...")
            async with ClientSession(read, write) as session:
                print("[DEBUG] Initializing...")
                await session.initialize()
                print("[DEBUG] ✓ Session initialized!")

                print("\n[DEBUG] Listing tools...")
                tools = await session.list_tools()
                print(f"[DEBUG] Got {len(tools.tools)} tools")

                print("\nAvailable tools:")
                for tool in tools.tools:
                    print(f"  - {tool.name}: {tool.description}")

                print("\n[DEBUG] Calling list_directory...")
                result = await session.call_tool(
                    "list_directory",
                    arguments={"path": "."},
                )

                print("\nTool result:")
                print(result)
    except Exception as e:
        print(f"[ERROR] {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
```